# ABCI-MI GPU End-to-End Pipeline

Run this notebook in Google Colab or Kaggle with a GPU runtime. It uses strict REAL mode, MOSS transcription, PyAnnote diarization, ACE processing, persistence, and report generation.

Before running: accept access to `OpenMOSS-Team/MOSS-Transcribe-Diarize` and `pyannote/speaker-diarization-3.1` on Hugging Face, then provide a Hugging Face token with access to both models.

In [ ]:
# GPU diagnostics
import os
import sys
import subprocess
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Torch CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime in Colab/Kaggle before continuing.')
print('GPU:', torch.cuda.get_device_name(0))
test = torch.ones((1024, 1024), device='cuda')
print('GPU tensor check:', test.sum().item())
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# Install the CUDA PyTorch stack and backend dependencies
%pip install -q --upgrade --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124
%pip install -q fastapi uvicorn sqlalchemy[asyncio] asyncpg psycopg2-binary alembic pydantic pydantic-settings python-dotenv redis qdrant-client httpx python-multipart soundfile psutil librosa transformers accelerate bitsandbytes pyannote.audio
print('Dependencies installed. Restart the runtime only if the kernel still reports CPU-only torch.')

## Configure the project

Set `PROJECT_ROOT` to the cloned or uploaded repository. For Kaggle, a common location is `/kaggle/working/abci-mi-backend-application`; for Colab, it is often `/content/abci-mi-backend-application`.

In [ ]:
from pathlib import Path
import gc
import os
import torch

PROJECT_ROOT = Path('/content/abci-mi-backend-application')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path('/kaggle/working/abci-mi-backend-application')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError('Clone or upload the ABCI-MI repository, then update PROJECT_ROOT.')

BACKEND_ROOT = PROJECT_ROOT / 'backend'
assert (BACKEND_ROOT / 'cli.py').exists(), f'Backend CLI not found under {BACKEND_ROOT}'
os.chdir(BACKEND_ROOT)
os.environ.update({
    'EXECUTION_MODE': 'REAL',
    'OPENMOSS_DEVICE': 'cuda',
    'AUDIO_CHUNK_DURATION_SECONDS': '60',
    'AUDIO_CHUNK_OVERLAP_SECONDS': '5',
    'AUDIO_CHUNK_THRESHOLD_SECONDS': '60',
    'AUDIO_CHUNK_CONCURRENCY': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})

# Memory policy: run one model family at a time.
RUN_PYANNOTE_SMOKE_TEST = True
RUN_FULL_ABCI_PIPELINE = False
MOSS_MAX_NEW_TOKENS = 512

def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

print('Project:', PROJECT_ROOT)
print('Backend:', BACKEND_ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# Securely provide the Hugging Face token without displaying it
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = getpass('Hugging Face token: ')
if not HF_TOKEN:
    raise ValueError('A Hugging Face token is required for REAL MOSS and PyAnnote inference.')
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ['HF_TOKEN'] = HF_TOKEN
print('Hugging Face authentication configured.')

In [ ]:
# Select a benchmark audio file
from pathlib import Path

BENCHMARK_DATASETS = {
    'AMI': 'AMI (headset mix, only_words)',
    'AISHELL-4': 'AISHELL-4',
    'AliMeeting': 'AliMeeting (channel 1)',
    'AVA-AVD': 'AVA-AVD',
    'DIHARD-3': 'DIHARD 3 (Full)',
    'MSDWILD': 'MSDWILD',
    'REPERE': 'REPERE (phase 2)',
    'VoxConverse': 'VoxConverse v0.3',
}
DATASET_NAME = 'AMI'
print('Selected benchmark:', BENCHMARK_DATASETS[DATASET_NAME])

AUDIO_PATH = PROJECT_ROOT / 'data' / 'audio' / 'raw' / 'ES2004a.Mix-Headset.wav'
if not AUDIO_PATH.exists():
    kaggle_matches = list(Path('/kaggle/input').glob('**/*.wav')) if Path('/kaggle/input').exists() else []
    if kaggle_matches:
        AUDIO_PATH = kaggle_matches[0]
if not AUDIO_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        AUDIO_PATH = Path(next(iter(uploaded)))
    except ImportError as exc:
        raise FileNotFoundError('Place a benchmark WAV under /kaggle/input or upload it in Colab, then rerun this cell.') from exc
if not AUDIO_PATH.exists():
    raise FileNotFoundError(f'Audio file not found: {AUDIO_PATH}')
print('Audio:', AUDIO_PATH)
print('Size MB:', round(AUDIO_PATH.stat().st_size / 1024 / 1024, 2))

## Optional benchmark data download

AMI signals are selected through the official [AMI download chooser](https://groups.inf.ed.ac.uk/ami/download/). Select the meeting and **Headset mix** signal, then point `AUDIO_PATH` at the downloaded WAV. AMI manual annotations can be downloaded automatically below. VoxConverse provides direct dev/test WAV archives and RTTM annotations through the linked repository.

In [ ]:
# Optional downloads from the official dataset sources
import shutil
import subprocess
import urllib.request
import zipfile

DOWNLOAD_BENCHMARK_DATA = False
VOX_SPLIT = 'dev'  # 'dev' or 'test'
DATA_ROOT = PROJECT_ROOT / 'benchmark_data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DOWNLOAD_BENCHMARK_DATA and DATASET_NAME == 'VoxConverse':
    vox_root = DATA_ROOT / 'voxconverse'
    vox_repo = vox_root / 'repo'
    vox_root.mkdir(parents=True, exist_ok=True)
    if not vox_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/joonson/voxconverse.git', str(vox_repo)], check=True)
    wav_zip = vox_root / f'voxconverse_{VOX_SPLIT}_wav.zip'
    wav_url = f'https://www.robots.ox.ac.uk/~vgg/data/voxconverse/data/voxconverse_{VOX_SPLIT}_wav.zip'
    if not wav_zip.exists():
        print('Downloading:', wav_url)
        urllib.request.urlretrieve(wav_url, wav_zip)
    wav_root = vox_root / f'{VOX_SPLIT}_wav'
    if not wav_root.exists():
        with zipfile.ZipFile(wav_zip) as archive:
            archive.extractall(wav_root)
    candidates = sorted(wav_root.glob('**/*.wav'))
    if not candidates:
        raise FileNotFoundError('No VoxConverse WAV files found after extraction.')
    AUDIO_PATH = candidates[0]
    RTTM_PATH = vox_repo / VOX_SPLIT / f'{AUDIO_PATH.stem}.rttm'
    print('VoxConverse audio:', AUDIO_PATH)
    print('VoxConverse reference RTTM:', RTTM_PATH if RTTM_PATH.exists() else 'not found')
elif DOWNLOAD_BENCHMARK_DATA and DATASET_NAME == 'AMI':
    annotations_zip = DATA_ROOT / 'ami_public_manual_1.6.2.zip'
    annotations_url = 'https://groups.inf.ed.ac.uk/ami/AMICorpusAnnotations/ami_public_manual_1.6.2.zip'
    if not annotations_zip.exists():
        print('Downloading AMI manual annotations:', annotations_url)
        urllib.request.urlretrieve(annotations_url, annotations_zip)
    print('AMI annotations saved to:', annotations_zip)
    print('Download the AMI Headset mix signal with the official chooser, then set AUDIO_PATH to its WAV.')
else:
    print('Benchmark download disabled. Using the selected local/uploaded audio:', AUDIO_PATH)

In [ ]:
# Run the project's strict REAL-mode prerequisite check
import subprocess

check = subprocess.run([sys.executable, 'cli.py', 'meeting', 'check-real-mode'], text=True)
if check.returncode != 0:
    raise RuntimeError('REAL-mode prerequisites failed. Check GPU visibility, model access, and Hugging Face permissions.')
print('REAL-mode prerequisites passed.')

In [ ]:
# Direct PyAnnote benchmark smoke test (one model at a time)
# PyAnnote does not use bitsandbytes; FP16 autocast reduces activation memory.
import gc
import torch
import soundfile as sf
from pyannote.audio import Pipeline

if RUN_PYANNOTE_SMOKE_TEST:
    waveform, sample_rate = sf.read(str(AUDIO_PATH), dtype='float32', always_2d=True)
    waveform = torch.from_numpy(waveform.T)
    pipeline = Pipeline.from_pretrained(
        'pyannote/speaker-diarization-3.1',
        token=HF_TOKEN,
    )
    if pipeline is None:
        raise RuntimeError('PyAnnote pipeline could not be loaded.')
    pipeline.to(torch.device('cuda'))
    with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.float16):
        diarization = pipeline({'waveform': waveform, 'sample_rate': sample_rate})

    rttm_path = Path('benchmark_diarization.rttm')
    with rttm_path.open('w', encoding='utf-8') as handle:
        diarization.write_rttm(handle)
    speakers = sorted({speaker for _, _, speaker in diarization.itertracks(yield_label=True)})
    turn_count = sum(1 for _ in diarization.itertracks(yield_label=True))
    print('Benchmark:', BENCHMARK_DATASETS[DATASET_NAME])
    print('Sample rate:', sample_rate)
    print('Speakers:', speakers)
    print('Speaker turns:', turn_count)
    print('RTTM output:', rttm_path.resolve())

    del diarization, pipeline, waveform
    clear_gpu_memory()
    print('PyAnnote released; GPU memory cleared before any MOSS run.')
else:
    print('PyAnnote smoke test disabled.')

In [ ]:
# Isolated MOSS transcription smoke test (4-bit CUDA quantization)
# OpenMOSSProvider uses bitsandbytes NF4 quantization when CUDA is available.
from app.ai.multilingual_asr import OpenMOSSProvider

RUN_MOSS_SMOKE_TEST = True
if RUN_MOSS_SMOKE_TEST:
    moss = OpenMOSSProvider({
        'model_id': 'OpenMOSS-Team/MOSS-Transcribe-Diarize',
        'device': 'cuda',
        'token': HF_TOKEN,
    })
    moss_segments = await moss.transcribe(str(AUDIO_PATH), options={'chunk_meta': None})
    print('MOSS segments:', len(moss_segments))
    for segment in moss_segments[:5]:
        print(segment.model_dump())
    del moss, moss_segments
    clear_gpu_memory()
    print('MOSS released; GPU memory cleared.')
else:
    print('MOSS smoke test disabled.')

In [ ]:
# Optional full ABCI-MI run
# MOSS is loaded by the backend with 4-bit bitsandbytes quantization on CUDA.
# Keep this disabled on a 15 GB limit: the full app may load MOSS and PyAnnote in one process.
import subprocess

if RUN_FULL_ABCI_PIPELINE:
    command = [sys.executable, 'cli.py', 'meeting', 'process-real', str(AUDIO_PATH)]
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print('--- stderr ---')
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Pipeline failed with exit code {result.returncode}. Review the output above.')
    print('REAL GPU pipeline completed successfully.')
else:
    print('Full ABCI-MI pipeline disabled. Run the isolated model tests one at a time.')

## Benchmark datasets and notes

The default sample is **AMI (headset mix, only_words)**: `ES2004a.Mix-Headset.wav`, matching the AMI benchmark row listed on the `pyannote/speaker-diarization-3.1` model card.

The model card also reports benchmarks for AISHELL-4, AliMeeting channel 1, AVA-AVD, DIHARD 3 Full, MSDWild, REPERE phase 2, and VoxConverse v0.3. To test another dataset, set `DATASET_NAME` and place its WAV under `/kaggle/input`, upload it in Colab, or update `AUDIO_PATH`. For quantitative DER evaluation, provide the matching reference RTTM and score the generated `benchmark_diarization.rttm` with the dataset's official protocol.

The notebook uses 60-second chunks because MOSS and PyAnnote can exceed the VRAM of smaller GPUs. On a 16 GB GPU, `AUDIO_CHUNK_DURATION_SECONDS` can be increased to `300` or `600`.

A Hugging Face `403 Forbidden` from PyAnnote means the account has not accepted the model terms or the token lacks access; it is not a CUDA failure.